## Split aleatório 70/30 por usuário

Cada usuário tem exatamente 30% das suas avaliações no teste e 70% no treino.  
Sem filtro de rating — usa o dataset completo (100 836 linhas).

In [1]:
import pandas as pd
import os

## 1. Carrega o dataset completo

In [2]:
df = pd.read_csv('ml-latest-small/ratings.csv')

print(f'Total de ratings : {df.shape[0]}')
print(f'Usuários únicos  : {df["userId"].nunique()}')
print(f'Filmes únicos    : {df["movieId"].nunique()}')

Total de ratings : 100836
Usuários únicos  : 610
Filmes únicos    : 9724


## 2. Split aleatório 70/30 por usuário

In [3]:
RANDOM_STATE = 42

# Para cada usuário, sorteia 30% dos índices para teste
test_idx = (
    df.groupby('userId', group_keys=False)
    .apply(lambda x: x.sample(frac=0.3, random_state=RANDOM_STATE))
    .index
)

df_test  = df.loc[test_idx]
df_train = df.drop(index=test_idx)

print(f'Treino : {df_train.shape[0]} linhas')
print(f'Teste  : {df_test.shape[0]} linhas')
print(f'Total  : {df_train.shape[0] + df_test.shape[0]} (esperado: {df.shape[0]})')

Treino : 70595 linhas
Teste  : 30241 linhas
Total  : 100836 (esperado: 100836)


## 3. Verificação da proporção por usuário

In [4]:
train_counts = df_train.groupby('userId').size().rename('train')
test_counts  = df_test.groupby('userId').size().rename('test')

stats = pd.concat([train_counts, test_counts], axis=1)
stats['total']    = stats['train'] + stats['test']
stats['test_pct'] = (stats['test'] / stats['total'] * 100).round(1)

print('Proporção de teste por usuário (estatísticas):')
print(stats['test_pct'].describe().round(2))
print(f'\nTodos os usuários presentes no treino: {df_train["userId"].nunique() == df["userId"].nunique()}')
print(f'Todos os usuários presentes no teste : {df_test["userId"].nunique() == df["userId"].nunique()}')

Proporção de teste por usuário (estatísticas):
count    610.00
mean      30.01
std        0.65
min       28.60
25%       29.70
50%       30.00
75%       30.30
max       32.00
Name: test_pct, dtype: float64

Todos os usuários presentes no treino: True
Todos os usuários presentes no teste : True


## 4. Salva os splits

In [ ]:
os.makedirs('data_spliting_hybrid', exist_ok=True)

df_train.to_csv('data_spliting_hybrid/training.csv', index=False)
df_test.to_csv('data_spliting_hybrid/testing.csv',   index=False)

print('Arquivos salvos em data_spliting_hybrid/')
print(f'  training.csv : {df_train.shape[0]} linhas')
print(f'  testing.csv  : {df_test.shape[0]} linhas')

Arquivos salvos em data_spliting_hybrid2/
  training.csv : 70595 linhas
  testing.csv  : 30241 linhas
